In [ ]:
import os 

import torch 
import torch.nn as nn 
import torch.optim as optim 
from torch.utils.data import DataLoader 
import torch.utils.data as data 
import torchvision.transforms as transforms 
import torch.nn.functional as F 
from pydicom import Dataset 
from PIL import Image 
from tqdm import tqdm 

from src.utils.dataloader import BreastMNISTLoader, BCDRLoader, BUSBRALoader, INbreastLoader, BUSUCLMLoader, BreastLesionsUSGLoader 
from src.models.main.models import(
QuantumModel,
ExtentedQuantumModel,
)
from src.utils.training import(
training_loop,
test_model,
)

import numpy as np 
import pandas as pd 
import seaborn as sns 
import matplotlib.pyplot as plt 
from sklearn.metrics import confusion_matrix,roc_curve,auc 

In [ ]:
device = torch.device("cuda"if torch.cuda.is_available()else "cpu")
if torch.cuda.is_available():
    print(f"Using: {torch.cuda.get_device_name(0)}")
    print(f"CUDA: {torch.version.cuda }")
else :
    print("CUDA is not available. Using CPU.")

In [ ]:
loader = BreastMNISTLoader(
batch_size = 16,
image_size = 28,
)

train_loader,test_loader,val_loader = loader.get_dataloaders()
dataloader = {"train":train_loader,"val":val_loader,"test":test_loader }

In [ ]:
n_qubits = 4 
n_layers = 1 
rand_params = np.random.uniform(high = 2 *np.pi,size =(n_layers,n_qubits))
num_classes = 2 
patch_size = 2 


sample_images,_ = next(iter(train_loader))
input_h,input_w = sample_images.shape [-2 ],sample_images.shape [-1 ]
output_size =(max(1,input_h //patch_size),max(1,input_w //patch_size))

print(f"Input size: {input_h }x{input_w } | Quantum output_size: {output_size }")

In [ ]:
model_basic = QuantumModel(
n_qubits = n_qubits,
n_layers = n_layers,
rand_params = rand_params,
output_size = output_size,
patch_size = patch_size,
num_classes = num_classes 
).to(device)

print(f"QuantumModel created with {sum(p.numel()for p in model_basic.parameters())} parameters")

In [ ]:
model_trainable = ExtentedQuantumModel(
n_qubits = n_qubits,
n_layers = n_layers,
num_classes = num_classes,
patch_size = patch_size,
mode = "2d",
stride = None,
use_parallel = True,
).to(device)

print(f"ExtentedQuantumModel created")

In [ ]:

model=model_basic 

print(f"\nSelected model: {model.__class__.__name__ }")
print(f"Total parameters: {sum(p.numel()for p in model.parameters())}")
print(f"Trainable parameters: {sum(p.numel()for p in model.parameters()if p.requires_grad)}")

In [ ]:

num_epochs=20 
learning_rate=3e-3 
save_dir="extra"
os.makedirs(save_dir,exist_ok = True)


if isinstance(model,(QuantumModel)):
    with torch.no_grad():
        dummy_input = torch.randn(1,1,input_h,input_w).to(device)
        _ = model(dummy_input)
    print("✓ Model FC layer initialized")


criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(
model.parameters(),
lr = learning_rate 
)

print(f"\nTraining configuration:")
print(f"  Epochs: {num_epochs }")
print(f"  Learning rate: {learning_rate }")
print(f"  Loss function: CrossEntropyLoss")
print(f"  Save directory: {save_dir }")

In [ ]:
history = training_loop(
model = model,
train_loader = train_loader,
val_loader = val_loader,
criterion = criterion,
optimizer = optimizer,
device = device,
num_epochs = num_epochs,
save_dir = save_dir,
save_best_only = True,
save_all_epochs = True,
)

print("\n✓ Training completed!")

In [ ]:
print("Initializing model layers...")
model.eval()

best_model_path = os.path.join(save_dir,"best_model.pth")
if os.path.exists(best_model_path):
    checkpoint = torch.load(best_model_path,map_location = device,weights_only = True)

    if isinstance(checkpoint,dict)and "model_state_dict"in checkpoint :
        state_dict = checkpoint ["model_state_dict"]
        epoch = checkpoint.get("epoch",None)
        val_acc = checkpoint.get("val_acc",None)
    else :
        state_dict = checkpoint 
        epoch = None 
        val_acc = None 

    model.load_state_dict(state_dict)
    print(f"✓ Loaded best model from {best_model_path }")
    print(f"  Epoch: {epoch if epoch is not None else 'N/A'}")
    print(f"  Val Accuracy: {val_acc :.4f}"if val_acc is not None else "  Val Accuracy: N/A")
else :
    print("⚠ Best model not found, using current weights")

hyperparams = {
"epochs":num_epochs,
"batch_size":16,
"learning_rate":learning_rate,
"n_qubits":n_qubits,
"n_layers":n_layers,
"patch_size":patch_size,
"model":model.__class__.__name__ 
}

test_loss, test_acc, test_prec, test_rec, test_f1, test_auc = test_model(
model = model,
dataloader = test_loader,
criterion = criterion,
device = device,
hyperparams = hyperparams,
output_file = "noencoder.txt",
extract_features = False,
compute_detailed_metrics = True 
)

print("\n"+"="*50)
print("FINAL TEST RESULTS")
print("="*50)
print(f"Test Loss:      {test_loss :.4f}")
print(f"Test Accuracy:  {test_acc :.4f}")
print(f"Test Precision: {test_prec :.4f}")
print(f"Test Recall:    {test_rec :.4f}")
print(f"Test F1 Score:  {test_f1 :.4f}")
print(f"Test AUC:       {test_auc :.4f}")
print("="*50)